# 21. Order statuses as columns, with zeros where there are none

**Track:** sql | **Difficulty:** easy | **Tags:** pivot, conditional aggregation, case when, nulls

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("021-pivot-order-statuses")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "orders": """
UEFSMRUEFeAaFc4NTBWsAxUAEgAAsA0IMXUABQEAMg0IADMNCAA0DQgANQ0IADYNCAA3DQgAOA0IADkNCAA6DQgAOw0IADwNCAA9
DQgAPg0IAD8NCABADQgAQQ0IAEINCABDDQgARA0IAEUNCABGDQgARw0IAEgNCABJDQgASg0IAEsNCABMDQgATQ0IAE4NCABPDQgA
UA0IAFENCABSDQgAUw0IAFQNCABVDQgAVg0IAFcNCABYDQgAWQ0IAFoNCABbDQgAXA0IAF0NCABeDQgAXw0IAGANCABhDQgAYg0I
AGMNCABkDQgAZQ0IAGYNCABnDQgAaA0IAGkNCABqDQgAaw0IAGwNCABtDQgAbg0IAG8NCABwDQgAcQ0IAHINCABzDQgAdA0IAHUN
CAB2DQgAdw0IAHgNCAB5DQgAeg0IAHsNCAB8DQgAfQ0IAH4NCAB/DQgAgA0IAIENCACCDQgAgw0IAIQNCACFDQgAhg0IAIcNCACI
DQgAiQ0IAIoNCACLDQgAjA0IAI0NCACODQgAjw0IAJANCACRDQgAkg0IAJMNCACUDQgAlQ0IAJYNCACXDQgAmA0IAJkNCACaDQgA
mw0IAJwNCACdDQgAng0IAJ8NCACgDQgAoQ0IAKINCACjDQgApA0IAKUNCACmDQgApw0IAKgNCACpDQgAqg0IAKsNCACsDQgArQ0I
AK4NCACvDQgAsA0IALENCACyDQgAsw0IALQNCAC1DQgAtg0IALcNCAC4DQgAuQ0IALoNCAC7DQgAvA0IAL0NCAC+DQgAvw0IAMAN
CADBDQgAwg0IAMMNCADEDQgAxQ0IAMYNCADHDQgAyA0IAMkNCADKDQgAyw0IAMwNCADNDQgAzg0IAM8NCADQDQgA0Q0IANINCADT
DQgA1A0IANUNCADWDQgA1w0IANgNCADZDQgA2g0IANsNCADcDQgA3Q0IAN4NCADfDQgA4A0IAOENCADiDQgA4w0IAOQNCADlDQgA
5g0IAOcNCADoDQgA6Q0IAOoNCADrDQgA7A0IAO0NCADuDQgA7w0IAPANCADxDQgA8g0IAPMNCAD0DQgA9Q0IAPYNCAD3DQgA+A0I
APkNCAD6DQgA+w0IAPwNCAD9DQgA/g0IAP8NCAgAdgAFAQABDQgAAg0IAAMNCAAEDQg8BXYAAAAAAAAGdgAAAAAAABUAFcIDFcoD
LBWsAxUQFQYVBhwYCAZ2AAAAAAAAGAgxdQAAAAAAABYAKAgGdgAAAAAAABgIMXUAAAAAAAAREQAAAOEB8OADAAAArAMBCDcAAQID
BAUGBwgJCgsMDQ4PEBESExQVFhcYGRobHB0eHyAhIiMkJSYnKCkqKywtLi8wMTIzNDU2Nzg5Ojs8PT4/QEFCQ0RFRkdISUpLTE1O
T1BRUlNUVVZXWFlaW1xdXl9gYWJjZGVmZ2hpamtsbW5vcHFyc3R1dnd4eXp7fH1+f4CBgoOEhYaHiImKi4yNjo+QkZKTlJWWl5iZ
mpucnZ6foKGio6SlpqeoqaqrrK2ur7CxsrO0tba3uLm6u7y9vr/AwcLDxMXGx8jJysvMzc7P0NHS09TVAAAVBBXAARV4TBUYFQAS
AABgBAEACQEABQkHBAAKDQgACQ0IAAgNCAAGDQgABA0IAAMNCAACDQgACw0IPAcAAAAAAAAADAAAAAAAAAAVABXqARXwASwVrAMV
EBUGFQYcGAgMAAAAAAAAABgIAQAAAAAAAAAWACgIDAAAAAAAAAAYCAEAAAAAAAAAEREAAAB18HQDAAAArAMBBDcQMkAlRHYUBIOH
qXeokKgSgQUokYIVejEwp7J3UxsRSzY4pycZkmUDVqBgWzcVsAuoBYBqGlEXqIhgIZU6UxA1EzUWNkGjp4QDQGVYa3QmSlJDMGKE
ZUNwoIKqSiCxKgRnhypDMmW1aQWBUAAVBBV8FX5MFQoVABIAAD4wCQAAAERFTElWRVJFRAENsENBTkNFTExFRAgAAABSRVRVUk5F
RAcAAABQRU5ESU5HCQAAAGRlbGl2ZXJlZBUAFeABFeIBLBWsAxUQFQYVBhw2FCgJZGVsaXZlcmVkGAlDQU5DRUxMRUQREQAAAHBE
HQAAAAO9EAED/ioBA/4WAQX+BQnwWE4BA/4yAQP8MgEDEgATAQAAQiJAWAYAAAIIgIAAQBAwAQBAApCAGIAEEAAfCQIAAzAAQwAg
ABAsgAYIAQAMwgIAAgAFBJAAQAIAWAAAmBAAEBBBA4IgBAAAFQQV0hAV3hBMFaoDFQASAACpCPQgAQAACe74AAAGq9AAAAs0XAAA
DU0UAAAJaMoAAAINlgAACCa4AAAHz84AAAqDmgAAClzuAAADfDAAAArQjgAABKeaAAAKdJAAAAfmdgAABY7aAAAIXAoAAAtZRgAA
BaWCAAAEi6wAAAdl8gAAAqNkAAAIEBAAAAVH9AAAAOn8AAAGbLoAAAIf8gAACqoUAAAJ51oAAApLKAAAA/3gAAAC9/YAAAXGIAAA
DUDGAAABjtQAAANWfgAAA6MOAAANlSYAAAteWgAAC3NAAAAMK9IAAAbIVAAABzyKAAAGSMoAAAtMlAAAB3JAAAAHGJoAAAhL1AAA
A8NIAAAGFZ4AAAQfeAAAAejeAAAKfoYAAAvzkgAAAdwsAAAKzpoAAADK7gAADDohDvD1GRoAAAoNogAABdFCAAAFyjoAAAsy/gAA
AQd6AAAItRoAAARuLgAAA36IAAAKJzgAAAJnCgAAA7JKAAAE2KAAAAbOxgAACHyoAAALq7IAAAi9sgAAB0JmAAAGnDAAAAwPgAAA
AZsiAAAF6IAAAAFPKAAABnwoAAAAz9AAAAZETAAADCQ0AAAFwNoAAAa1xgAABH+QAAAEZfoAAAMBiAAACBpqAAAKOAQAAAI0EAAA
AtniAAADVFgAAAj7zgAACZE4AAAM89IAAACeygAADI7YAAALJCYAAAvL7AAACzDYAAANWP4AAAxkEgAACzFuAAAIXDwAAAJ4Aeb0
CQKSgAAAC+vCAAAItngAAAypNgAABNmaAAAFm1oAAAbDDgAAB0tiAAAAjpQAAAJxZAAACna2AAAGT9IAAAyS8gAAALvkAAAHqIwA
AAppoAAAB/VOAAAMInIAAArw+gAACRl+AAAHE4YAAAaWVAAAA1GcAAAIMOAAAA2zCAAABReEAAAMaMIAAA1rWgAAAtEYAAAHtXAA
AAlTHAAAB9wcAAANqBgAAAcpNAAAAarCAAADdb4AAALd/AAAB2l2AAAMRZoAAAjSNAAAAZZyAAAMPygAAAHv5gAADJicAAABpLQA
AAQIngAAAm6oAAADQ74AAAXyEgAAC5EiAAAHvKoAAAfwCAAAAsiAAAAFlFIAAAJVRAAABkyAAAACLdAAAABeugAAB5VoAAAGFy4A
AAdfTgAAAhU0AAAG6rQAAArucAAABexoAAAG8bwAAAIWYAAABHOmAAAHhM4AAANpogAAB+gGAAAHHB4AAAs/TAAAAU0CAAAIXG4A
AADy+AAABgaUAAAHClgAAASvBgAADUcGAAAHQdAAAAszYgAAApWGAAAJAH4AAAiivgAAAurgAAAIjHoAAAywogAAClCgAAALrEgA
AAgq0gAACGJ8AAAKaHQAAAvakgAAAaicAAANQ+YAAAzEKgAAAZfQAAAG+rgAAAeBSgAADGuwAAABBhwAAARDNgAAAKVuAAACojgV
ABXCAxXKAywVrAMVEBUGFQYcGAUAAA2zCBgFAAAAXroWACgFAAANswgYBQAAAF66EREAAADhAfDgAwAAAKwDAQg3AAECAwQFBgcI
CQoLDA0ODxAREhMUFRYXGBkaGxwdHh8gISIjJCUmJygpKissLS4vMDEyMzQ1Njc4OTo7PD0+P0BBQkNERUZHSElKS0xNTk9QUVJT
VFVWV1hZWltcXV5fYGFiY2RlZmdoaWprbG1ub3BxcnN0dXZ3eHl6e3x9fn+AgYKDhIWGh4iJiouMjY6PkJGSk5SVlpeYmZqbnJ2e
n6ChoqOkpZ2mp6ipqqusra6vsLGys7S1tre4ubq7vL2+v8DBwsPExcbHyMnKy8zNzs/Q0dLT1AAAFQQZXDUAGAZzY2hlbWEVCAAV
BCUCGAhvcmRlcl9pZAAVBCUCGAhzdG9yZV9pZAAVDCUCGAZzdGF0dXMlAEwcAAAAFQ4VChUCGAZhbW91bnQlChUEFRQsXBUEFRQA
AAAWrAMZHBlMJgAcFQQZNQAGEBkYCG9yZGVyX2lkFQIWrAMWyB8WvhIm+A0mCBwYCAZ2AAAAAAAAGAgxdQAAAAAAABYAKAgGdgAA
AAAAABgIMXUAAAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYArAMAAAAmABwVBBk1AAYQGRgIc3RvcmVfaWQVAhasAxbMBBaK
BCbcEybGEhwYCAwAAAAAAAAAGAgBAAAAAAAAABYAKAgMAAAAAAAAABgIAQAAAAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYA
rAMAAAAmABwVDBk1AAYQGRgGc3RhdHVzFQIWrAMW2AMW3AMm6hcm0BYcNhQoCWRlbGl2ZXJlZBgJQ0FOQ0VMTEVEEREAGSwVBBUA
FQIAFQAVEBUCADwWhBwZBhkmFJgDAAAAJgAcFQ4ZNQAGEBkYBmFtb3VudBUCFqwDFqIVFrYVJqwrJqwaHBgFAAANswgYBQAAAF66
FgAoBQAADbMIGAUAAABeuhERABksFQQVABUCABUAFRAVAgA8KQYZJgCsAwAAABaOPRasAyYIFtovABkcGAxBUlJPVzpzY2hlbWEY
mAMvLy8vL3lnQkFBQVFBQUFBQUFBS0FBd0FCZ0FGQUFnQUNnQUFBQUFCQkFBTUFBQUFDQUFJQUFBQUJBQUlBQUFBQkFBQUFBUUFB
QUM0QUFBQWNBQUFBRUFBQUFBRUFBQUFhUC8vL3dBQUFRY1FBQUFBSUFBQUFBUUFBQUFBQUFBQUJnQUFBR0Z0YjNWdWRBQUFDQUFN
QUFRQUNBQUlBQUFBQ2dBQUFBSUFBQUNnLy8vL0FBQUJCUkFBQUFBY0FBQUFCQUFBQUFBQUFBQUdBQUFBYzNSaGRIVnpBQUFFQUFR
QUJBQUFBTXovLy84QUFBRUNFQUFBQUJ3QUFBQUVBQUFBQUFBQUFBZ0FBQUJ6ZEc5eVpWOXBaQUFBQUFDOC8vLy9BQUFBQVVBQUFB
QVFBQlFBQ0FBR0FBY0FEQUFBQUJBQUVBQUFBQUFBQVFJUUFBQUFKQUFBQUFRQUFBQUFBQUFBQ0FBQUFHOXlaR1Z5WDJsa0FBQUFB
QWdBREFBSUFBY0FDQUFBQUFBQUFBRkFBQUFBQUFBQUFBPT0AGCBwYXJxdWV0LWNwcC1hcnJvdyB2ZXJzaW9uIDI1LjAuMRlMHAAA
HAAAHAAAHAAAAOsDAABQQVIx
""",
}

_data_dir = pathlib.Path("data") / "021-pivot-order-statuses"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

orders = spark.read.parquet((_data_dir / "orders.parquet").resolve().as_posix())
orders.createOrReplaceTempView("orders")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'cancelled_orders': 'c2f67f3f618c3f4401807762e8893ee261133c60e1d76c334a5b114c182433b7',
                         'delivered_amount': '867ae8cbadf111275604342923d701f9d46bc81b2081ed8c578f3ec358563979',
                         'delivered_orders': '84b580cd872df2689f49f3ea0f84cc51ca8f5dd62a3dfb0ae894de9b4ac1cdb2',
                         'other_orders': '44b92ea3efa6e7e6cdaaaf285cc55e242bc204ea470eb51a5b52d679c5ab78f0',
                         'returned_orders': 'bcfba81376959eb1430c2099a5c593da5e75683271fda005f468a0c6e589e8fb',
                         'store_id': 'aa6aba41b0ecab8a55e5fcf1ac771c2df962b87e376e3873a4c7bc991e120fc5'},
 'columns': [['cancelled_orders', 'bigint'],
             ['delivered_amount', 'decimal(12,2)'],
             ['delivered_orders', 'bigint'],
             ['other_orders', 'bigint'],
             ['returned_orders', 'bigint'],
             ['store_id', 'bigint']],
 'fingerprint': 'd1a41b2cae9d3d16699dd1c14e8b1b9b62a9640ea534b0542bce5d2afac1fdcd',
 'order_matters': False,
 'row_count': 12}

check = make_check(EXPECTED)

## The situation

The operations team reads a daily sheet with one row per store and one column per
order status. The orders table has it the other way round: one row per order, with
the status in a column.

The first version of the query left blanks in the sheet for stores that had no
returns, and its columns did not add up to the number of orders.

## Input tables

### `orders`

One row per order.

| Column | Type | Description |
|---|---|---|
| `order_id` | BIGINT | Unique per order. |
| `store_id` | BIGINT | The store that took the order. |
| `status` | STRING | Normally `DELIVERED`, `CANCELLED` or `RETURNED`. Can be NULL, and can be something else. |
| `amount` | DECIMAL(10,2) | Order value in rupees. |

## Requirement

Return one row per store.

- `delivered_orders`, `cancelled_orders` and `returned_orders` count the orders whose
  status is exactly `DELIVERED`, `CANCELLED` or `RETURNED`.
- `other_orders` counts every other order: a NULL status, and any status that is not
  one of those three exact values (`delivered` in lowercase is one of these).
- The four counts of a store add up to its number of orders.
- `delivered_amount` is the total amount of the store's `DELIVERED` orders.
- A store with no orders of a kind shows 0 in that column, and 0.00 for the amount.
  Never NULL.

## Expected output

One row per store.

| Column | Type | Description |
|---|---|---|
| `store_id` | BIGINT | |
| `delivered_orders` | BIGINT | |
| `cancelled_orders` | BIGINT | |
| `returned_orders` | BIGINT | |
| `other_orders` | BIGINT | NULL and unrecognised statuses. |
| `delivered_amount` | DECIMAL(12,2) | 0.00 if the store has no delivered orders. |

In [ ]:
orders.show(5)

## Your answer

Write one Spark SQL query. The tables are available as views: `orders`.

In [ ]:
query = """
-- Write your Spark SQL query here.
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)